# Simple Ranker

This notebook walks through the three ranking ideas built in `src/ranking_service/impl_handrolled.py`.

In [1]:
import html
import json
import math
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import HTML, display

In [2]:
# Helper Functions

def show(stories, n=5, extra=None):
    """Render the first n stories as an HTML table. Reused at every step below
    instead of print() so ranked output is easy to scan.

    `extra` is an optional {column_name: fn(story) -> value} dict for
    showing a computed value (e.g. a score) alongside each story.
    """
    extra = extra or {}
    cols = list(extra.keys()) + ['points', 'title', 'author', 'comments']
    header = '<tr>' + ''.join(f'<th style="padding:4px 10px;text-align:left">{c}</th>' for c in cols) + '</tr>'
    rows = []
    for s in stories[:n]:
        cells = []
        for fn in extra.values():
            val = fn(s)
            cells.append(f'{val:.4f}' if isinstance(val, float) else str(val))
        cells += [
            str(s.get('points')),
            html.escape((s.get('title') or '')[:80]),
            html.escape(s.get('author') or ''),
            str(s.get('num_comments')),
        ]
        rows.append('<tr>' + ''.join(f'<td style="padding:4px 10px">{c}</td>' for c in cells) + '</tr>')
    display(HTML(f'<table>{header}{"".join(rows)}</table>'))

### Fetch data from HackerNews

The input data is committed to the repo so this notebook is reproducible
regardless of when you're watching at `data/hn_stories_snapshot.jsonl`. It was generated once with `data/fetch_live.py`, which
pages through the HN Algolia `search_by_date` API. Algolia caps pagination at 1000 results
per query, so the script windows by `created_at_i` to pull the full ~5K stories.

Not re-run here — uncomment to refresh the snapshot with the latest stories.

In [3]:
# import subprocess
# subprocess.run(
#     ['python3', '../../../data/fetch_live.py', '--n', '5000'],
#     stdout=open('../../../data/hn_stories_snapshot.jsonl', 'w'),
#     check=True,
# )

### Read downloaded HN data

Lets load and review the input data from at `data/hn_stories_snapshot.jsonl`.

In [4]:
# Load the snapshot
snapshot_path = Path('../../../data/hn_stories_snapshot.jsonl')
stories = [json.loads(line) for line in snapshot_path.read_text().splitlines() if line.strip()]

_times = []
for s in stories:
    try:
        if s.get('created_at'):
            _times.append(datetime.fromisoformat(s['created_at'].replace('Z', '+00:00')))
    except (ValueError, AttributeError):
        pass
DATA_NOW = max(_times) if _times else datetime.now(timezone.utc)

show(stories, n=10)


points,title,author,comments
1,India's Unconvincing Economic Facade,eatonphil,0
1,Russian politician says GTA 6 carries 'the stench of Americanism',HelloUsername,0
1,"Mu: A social app with a backbone, built on European values",modinfo,0
1,Eclipsa Video: HDR That Looks Right on Every Screen,ledoge,0
3,Claude Sonnet 5 System Card,adocomplete,0
1,Who can take on web exploit scans? The Banhammer™ can!,nate-gehringer,0
1,The DLL that was not present in memory despite not being formally unloaded 2,supermatou,0
1,Valve explains the Steam Machine (and that pricing) [video],HelloUsername,0
3,Ask HN: Help my web browser project be better,roschdal,0
1,AgentWire: Orchestrating many Claude Code sessions via tmux and voice-control,AdamTSaunders,0


### Understanding the data before we rank it

A few questions worth answering before building a ranker:
- **Points**: how skewed is it? If most stories sit at 1-2 points, popularity ranking is mostly noise outside a small head.
- **Comments**: same question — does engagement track points, or diverge?
- **Freshness**: how old is the snapshot? This determines how much the age term in the HN decay formula will actually matter.
- **Source concentration**: is one domain (e.g. github.com) dominating submissions? Relevant later for the Diversity/Integrity filter in the Ranker step.

In [5]:
def bucket_counts(values, edges):
    """Bucket numeric values into ranges defined by ascending `edges`.
    edges=[1, 2, 6, 11] produces buckets '1', '2-5', '6-10', '11+'.
    """
    labels = [str(edges[i]) if edges[i] == edges[i + 1] - 1 else f'{edges[i]}-{edges[i + 1] - 1}'
              for i in range(len(edges) - 1)]
    labels.append(f'{edges[-1]}+')
    counts = [0] * len(labels)
    for v in values:
        idx = next((i for i in range(len(edges) - 1) if edges[i] <= v < edges[i + 1]), len(labels) - 1)
        counts[idx] += 1
    return list(zip(labels, counts))


def bar_chart(rows, title=None, bar_width=300):
    """Render (label, count) pairs as a simple HTML horizontal bar chart."""
    max_count = max((c for _, c in rows), default=1) or 1
    bars = []
    for label, count in rows:
        px = int(bar_width * count / max_count)
        bars.append(
            f'<tr><td style="padding:2px 10px;white-space:nowrap">{html.escape(str(label))}</td>'
            f'<td style="padding:2px 0"><div style="background:#4C8BF5;height:14px;width:{px}px"></div></td>'
            f'<td style="padding:2px 10px">{count}</td></tr>'
        )
    heading = f'<h4 style="margin-bottom:4px">{html.escape(title)}</h4>' if title else ''
    display(HTML(f'{heading}<table>{"".join(bars)}</table>'))

In [6]:
bar_chart(
    bucket_counts([s.get('points') or 0 for s in stories], [1, 2, 6, 11, 26, 51, 101, 251, 501]),
    title='Points distribution',
)

1,,836
2-5,,3100
6-10,,463
11-25,,212
26-50,,99
51-100,,96
101-250,,115
251-500,,44
501+,,35


In [7]:
bar_chart(
    bucket_counts([s.get('num_comments') or 0 for s in stories], [0, 1, 3, 6, 11, 26, 51, 101]),
    title='Comments distribution',
)

0,,3143
1-2,,1100
3-5,,278
6-10,,112
11-25,,102
26-50,,67
51-100,,88
101+,,110


In [8]:
def age_hours(story):
    ts = datetime.fromisoformat(story['created_at'].replace('Z', '+00:00'))
    return (DATA_NOW - ts).total_seconds() / 3600

bar_chart(
    bucket_counts([age_hours(s) for s in stories], [0, 24, 48, 72, 96, 120, 144]),
    title='Story age (hours since snapshot was pulled)',
)


0-23,,1123
24-47,,945
48-71,,712
72-95,,779
96-119,,1044
120-143,,397
144+,,0


In [9]:
def bucket_means(items, value_fn, bucket_fn, edges):
    """Mean of value_fn(item) grouped by bucket_fn(item) into ranges defined by `edges`."""
    labels = [str(edges[i]) if edges[i] == edges[i + 1] - 1 else f'{edges[i]}-{edges[i + 1] - 1}'
              for i in range(len(edges) - 1)]
    labels.append(f'{edges[-1]}+')
    sums = [0.0] * len(labels)
    counts = [0] * len(labels)
    for item in items:
        idx = next((i for i in range(len(edges) - 1) if edges[i] <= bucket_fn(item) < edges[i + 1]), len(labels) - 1)
        sums[idx] += value_fn(item)
        counts[idx] += 1
    return [(label, round(sums[i] / counts[i], 1) if counts[i] else 0) for i, label in enumerate(labels)]

bar_chart(
    bucket_means(stories, lambda s: s.get('points') or 0, age_hours, [0, 24, 48, 72, 96, 120, 144]),
    title='Mean points by story age (hours)',
)

0-23,,9.3
24-47,,19.1
48-71,,21.9
72-95,,21.1
96-119,,16.4
120-143,,25.8
144+,,0


**Finding**: mean points jump from the 0-23h bucket to the 24-47h bucket, then roughly
plateau. This isn't "older stories accumulate more points forever" — it's that a story's
voting window is short (mostly the first ~1-2 days on the front page), so the 0-23h bucket
is artificially low: it's a mix of stories still actively gaining votes and stories that
will never get more. This is exactly why the HN formula divides by `age_hours` instead of
ignoring it — without decay, the ranking would have no way to surface anything new once
a backlog of older, already-voted-up stories exists.

In [10]:
from collections import Counter
from urllib.parse import urlparse

domains = Counter(
    urlparse(s['url']).netloc.removeprefix('www.')
    for s in stories if s.get('url')
)
no_url = sum(1 for s in stories if not s.get('url'))

bar_chart(domains.most_common(10), title='Top 10 source domains')
print(f'{no_url} stories ({no_url / len(stories):.1%}) have no url — these are text posts (e.g. "Ask HN").')

github.com,,511
youtube.com,,140
twitter.com,,62
medium.com,,53
en.wikipedia.org,,51
reuters.com,,48
wsj.com,,41
arxiv.org,,41
theguardian.com,,40
nytimes.com,,38


206 stories (4.1%) have no url — these are text posts (e.g. "Ask HN").


In [11]:
authors = Counter(s.get('author') for s in stories)
print(f'{len(authors)} unique authors across {len(stories)} stories')
bar_chart(authors.most_common(10), title='Top 10 submitters')

2921 unique authors across 5000 stories


speckx,,79
mooreds,,74
surprisetalk,,55
tosh,,53
Brajeshwar,,48
Bender,,42
rbanffy,,37
paulpauper,,36
1vuio0pswjnm7,,34
bookofjoe,,33


## Idea 1 — Rank by points (upvotes)

In [12]:
ranked = sorted(stories, key=lambda s: s.get('points') or 0, reverse=True)
show(ranked)

points,title,author,comments
1708,An entire Herculaneum scroll has been read for the first time,verditelabs,366
1348,Om Malik has died,minimaxir,171
1182,U.S. government will decide who gets to use GPT-5.6,alain94040,1233
1150,"The 'papers, please' era of the internet will decimate your privacy",bilsbie,615
1133,Previewing GPT‑5.6 Sol: a next-generation model,minimaxir,742


### Sanity check — does this match the real HN front page?

Each story in the snapshot carries `is_front_page`, derived from whether the HN API ever
tagged it `front_page` — i.e. whether it actually got promoted by HN's real ranking. We can
use this as a rough ground-truth check: how much does our top-K overlap with what HN
actually picked?

In [13]:
def front_page_match(ranked, k=30):
    """What fraction of our top-k actually reached HN's real front page?"""
    top_k = ranked[:k]
    matched = sum(1 for s in top_k if s.get('is_front_page'))
    total_fp = sum(1 for s in stories if s.get('is_front_page'))
    print(f'{matched}/{k} of our top-{k} actually reached the real HN front page ({matched / k:.0%}).')
    print(f'(For reference: {total_fp}/{len(stories)} stories in this snapshot ever reached the front page.)')

front_page_match(ranked)

5/30 of our top-30 actually reached the real HN front page (17%).
(For reference: 31/5000 stories in this snapshot ever reached the front page.)


**Finding**: only 5/30 (17%) of our raw-points top-30 actually reached the real front page.
Makes sense — points alone has no sense of time. A story from 4 days ago with 600 points
ranks above a 1-hour-old story with 80 points, even though the older one's voting window
likely closed days ago and the newer one is still climbing. Raw points rewards accumulated
totals, not "is this worth showing right now" — which is exactly the gap Idea 3's decay
formula is meant to close.

## Idea 2 — Rank by cohort (synthetic continent)

The original idea was to rank separately per user language or country — e.g. show
French-speaking users a French-flavored front page. Our dataset has neither signal: HN
titles are almost entirely English, and Algolia's `_tags` only distinguish `story` /
`job` / `poll` / `ask_hn` / `show_hn` / `front_page`, nothing demographic.

To still demonstrate the *mechanism* — same candidate pool, ranked separately per cohort —
we fabricate a `continent` field per story with a seeded random assignment. **This is
synthetic data for demonstration only, not a real signal**, so the front-page sanity check
from Idea 1 doesn't apply here: real HN has no concept of continent-specific front pages.

In [14]:
import random

CONTINENTS = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']

random.seed(42)
for s in stories:
    s['continent'] = random.choice(CONTINENTS)

bar_chart(
    Counter(s['continent'] for s in stories).most_common(),
    title='Synthetic continent assignment (should be roughly even — it\'s random)',
)

Oceania,,865
South America,,852
Asia,,842
Africa,,819
Europe,,814
North America,,808


In [15]:
# Same candidate pool, filtered to one cohort, then ranked the same way as Idea 1.
# Each continent gets its own top stories — a different front page per cohort.
for continent in CONTINENTS:
    cohort = [s for s in stories if s['continent'] == continent]
    ranked_cohort = sorted(cohort, key=lambda s: s.get('points') or 0, reverse=True)
    display(HTML(f'<h4>{continent} ({len(cohort)} stories)</h4>'))
    show(ranked_cohort, n=3)

points,title,author,comments
1093,Qwen 3.6 27B is the sweet spot for local development,stared,696
1093,GLM 5.2 beats Claude in our benchmarks,jms703,504
993,HackerRank open sourced its ATS. My resume scored 90/100. Oh wait 74. No – 88,sambellll,424


points,title,author,comments
637,The KIDS Act would require age checks to get online,bilsbie,563
624,.self: A new top-level domain designed to support self-hosting,HumanCCF,354
513,We can still stop California's 3D printer surveillance scheme,hn_acker,188


points,title,author,comments
1348,Om Malik has died,minimaxir,171
1182,U.S. government will decide who gets to use GPT-5.6,alain94040,1233
1150,"The 'papers, please' era of the internet will decimate your privacy",bilsbie,615


points,title,author,comments
995,Age verification is just a precursor to automated attribution of speech,arkhiver,611
843,"Apple raises prices of MacBooks, iPads",virgildotcodes,1252
728,EU to legislate about Chat Control behind closed doors,NeutralForest,428


points,title,author,comments
902,Pollen tried to remove my article and Google is assisting with it,taubek,126
812,Show HN: I made Google Trends for Hacker News by indexing 18 years of comments,ytkimirti,156
780,Zuckerberg's war on whistleblowers,HotGarbage,294


points,title,author,comments
1708,An entire Herculaneum scroll has been read for the first time,verditelabs,366
945,Anonymous GitHub account mass-dropping undisclosed 0-days,binyu,382
792,DSpark: Speculative decoding accelerates LLM inference [pdf],aurenvale,360


## Idea 3 — HN decay formula

```
score = (points - 1) / (age_hours + 2) ^ 1.8
```

The `(P-1)` removes the submitter's automatic upvote.

In [16]:
def hn_score(story):
    points = story.get('points') or 1
    created_at = story.get('created_at', '')
    try:
        ts = datetime.fromisoformat(created_at.replace('Z', '+00:00'))
        age_h = (DATA_NOW - ts).total_seconds() / 3600
    except Exception:
        age_h = 24
    return (points - 1) / (age_h + 2) ** 1.8

ranked_hn = sorted(stories, key=hn_score, reverse=True)
show(ranked_hn, extra={'score': hn_score})


score,points,title,author,comments
42.0696,566,Claude Code Is Steganographically Marking Requests,kirushik,174
24.8406,357,The labor share of income in the US is at its lowest post-war level,loughnane,337
12.9955,150,County with 37 Data Centers Asks Schools to 'Conserve Electricity',01-_-,83
11.0281,114,"EU commissioners shut down air conditioning for employees, leave theirs on",spwa4,92
10.8394,609,European digital ID wallets rely on safety services of Google and Apple,donohoe,258


### Sanity check — front page match (Idea 3)

This is the same formula HN itself uses, so we'd expect much higher overlap than Idea 1's
raw-points ranking. Let's check.

In [17]:
front_page_match(ranked_hn)

19/30 of our top-30 actually reached the real HN front page (63%).
(For reference: 31/5000 stories in this snapshot ever reached the front page.)


**Finding**: 19/30 (63%) matched the real front page — much better than Idea 1's 17%. The
remaining 11 miss for two reasons:

1. **Wall-clock-time bias.** We score all 5000 stories against "now," but HN's decay
   formula naturally limits the competitive pool to the last ~24–48 hours of submissions.
   Applying it to a 5-day backlog artificially inflates fresh stories that haven't had time
   to earn `front_page` yet, while the stories HN actually promoted days ago look stale.

2. **HN filters beyond the formula.** User flags, moderator penalties, flamewar detection,
   and suspicious-vote discounting all adjust a story's effective score invisibly. The
   decay formula is the input, not the final word — which is exactly what the Ranker step's
   integrity layer addresses.

## What we covered

| Idea | Key insight | Front-page match (top 30) |
|---|---|---|
| 1 — Raw points | Accumulated totals reward old stories regardless of age | 17% |
| 2 — Cohort ranking | Same formula per group — the mechanism works, the signal is synthetic | — |
| 3 — HN decay formula | `(points - 1) / (age + 2)^1.8` penalises stale stories | **63%** |

The decay formula reaches 63% because it's the same formula HN uses. The 37% gap is
HN's invisible layer — vote-manipulation detection, moderator penalties, flamewar
flags — that sits on top of the formula and isn't publicly documented.

**The remaining problem**: every user sees the same list. Alice (ML researcher) and
Bob (frontend developer) get identical front pages because the ranker reads only
`points` and `age` — the `user` argument exists but is never used.

**The Candidate Generators step** fixes the first half: before the ranker runs, four candidate generators
narrow the 5,000-story pool down to a few hundred stories that are plausibly relevant
*for this specific user*. The ranker formula stays the same — what changes is who gets
considered.